<a href="https://colab.research.google.com/github/25f3001314-dev/statistella-/blob/main/9_71.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import xgboost as xgb
import lightgbm as lgb # LightGBM was not used in the Ensemble but was part of model development.
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import re

# --- 0. Load Data ---
# Ensure 'train.csv' and 'test.csv' files are available in your /content/ directory.
# If they are not already loaded, load them here.
try:
    train_df = train_df
    test_df = test_df
except NameError:
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

# --- 1. Advanced Text Cleaning & Mega Text Creation ---
def clean_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
    return text

text_cols = ['Headline', 'Lead Types', 'Power Mentions', 'Agencies', 'Reasoning', 'Key Insights', 'Tags']
for col in text_cols:
    train_df[col] = train_df[col].fillna('missing').apply(clean_text)
    test_df[col] = test_df[col].fillna('missing').apply(clean_text)

train_df['mega_text'] = train_df[text_cols].astype(str).agg(' '.join, axis=1)
test_df['mega_text'] = test_df[text_cols].astype(str).agg(' '.join, axis=1)

# --- 2. TF-IDF Vectorization ---
vec = TfidfVectorizer(max_features=5000, ngram_range=(1,3), analyzer='char_wb', max_df=0.95, min_df=2, sublinear_tf=True)
X_train_tfidf = vec.fit_transform(train_df['mega_text']).toarray()
X_test_tfidf = vec.transform(test_df['mega_text']).toarray()

# --- 3. Extra Numeric Features ---
for col in text_cols:
    train_df[f'{col}_len'] = train_df[col].str.len()
    test_df[f'{col}_len'] = test_df[col].str.len()

extra_feats = [f'{col}_len' for col in text_cols]
X_train_extra = train_df[extra_feats].fillna(0).values
X_test_extra = test_df[extra_feats].fillna(0).values

# Combine TF-IDF with Extra Features
X_train_final = np.hstack([X_train_tfidf, X_train_extra]) # 5000 + 7 = 5007 features
X_test_final = np.hstack([X_test_tfidf, X_test_extra])   # 5000 + 7 = 5007 features

# Target Variable
y = train_df['Importance Score']

# --- 4. Train-Test Split ---
# For model evaluation
Xtr, Xte, ytr, yte = train_test_split(X_train_final, y, test_size=0.2, random_state=42)

# --- 5. Train the First XGBoost Model ---
# This model gave the lowest RMSE (approx 4.6739)
model_xgb = xgb.XGBRegressor(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
print("First XGBoost model (model_xgb) training is starting...")
model_xgb.fit(Xtr, ytr)
preds_xgb_eval = model_xgb.predict(Xte)
rmse_xgb_eval = np.sqrt(mean_squared_error(yte, preds_xgb_eval))
print(f"\nXGBoost (model_xgb) RMSE (on eval set): {rmse_xgb_eval:.4f}")

# --- 6. Train the Second XGBoost Model (Anti-Overfit) ---
# This was tuned to reduce overfitting
model_anti_over = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.03, max_depth=4,  # Conservative
    subsample=0.75, colsample_bytree=0.75,             # More random
    reg_alpha=0.8, reg_lambda=3.0,                     # Strong reg
    random_state=42, n_jobs=-1
)
print("Anti-Overfit model (model_anti_over) training is starting...")
model_anti_over.fit(Xtr, ytr, eval_set=[(Xte, yte)], verbose=0)
anti_rmse = np.sqrt(mean_squared_error(yte, model_anti_over.predict(Xte)))
print(f"Anti-Overfit XGBoost RMSE (on eval set): {anti_rmse:.4f}")

# --- 7. Ensemble Predictions and RMSE ---
# Ensemble RMSE on evaluation set
final_pred_eval = 0.7 * model_xgb.predict(Xte) + 0.3 * model_anti_over.predict(Xte)
ensemble_rmse_eval = np.sqrt(mean_squared_error(yte, final_pred_eval))
print(f"Ensemble RMSE (on eval set - 0.7 * model_xgb + 0.3 * model_anti_over): {ensemble_rmse_eval:.4f}")

# Final Ensemble Predictions on FULL TEST DATA
xgb_preds_final = model_xgb.predict(X_test_final)
anti_over_preds_final = model_anti_over.predict(X_test_final)
Final_Pred_Submission = 0.7 * xgb_preds_final + 0.3 * anti_over_preds_final

# --- 8. Create Submission File ---
submission_df = pd.DataFrame({'id': test_df['id'], 'Importance Score': Final_Pred_Submission})

# Ensure 'Importance Score' is numeric and handle non-finite values
submission_df['Importance Score'] = pd.to_numeric(submission_df['Importance Score'], errors='coerce')
submission_df['Importance Score'] = submission_df['Importance Score'].fillna(submission_df['Importance Score'].mean())

# Clip negative scores to 0 and max to 92
submission_df['Importance Score'] = submission_df['Importance Score'].apply(lambda x: max(0, x)).clip(0, 92)

submission_df.to_csv('final_top_ensemble_submission.csv', index=False)
print("\nDone! 'final_top_ensemble_submission.csv' is ready for download.")

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [2]:
import xgboost as xgb
import lightgbm as lgb # LightGBM was not used in the Ensemble but was part of model development.
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import re

# --- 0. Load Data ---
# Ensure 'train.csv' and 'test.csv' files are available in your /content/ directory.
# If they are not already loaded, load them here.
try:
    train_df = train_df
    test_df = test_df
except NameError:
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

# --- 1. Advanced Text Cleaning & Mega Text Creation ---
def clean_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
    return text

text_cols = ['Headline', 'Lead Types', 'Power Mentions', 'Agencies', 'Reasoning', 'Key Insights', 'Tags']
for col in text_cols:
    train_df[col] = train_df[col].fillna('missing').apply(clean_text)
    test_df[col] = test_df[col].fillna('missing').apply(clean_text)

train_df['mega_text'] = train_df[text_cols].astype(str).agg(' '.join, axis=1)
test_df['mega_text'] = test_df[text_cols].astype(str).agg(' '.join, axis=1)

# --- 2. TF-IDF Vectorization ---
vec = TfidfVectorizer(max_features=5000, ngram_range=(1,3), analyzer='char_wb', max_df=0.95, min_df=2, sublinear_tf=True)
X_train_tfidf = vec.fit_transform(train_df['mega_text']).toarray()
X_test_tfidf = vec.transform(test_df['mega_text']).toarray()

# --- 3. Extra Numeric Features ---
for col in text_cols:
    train_df[f'{col}_len'] = train_df[col].str.len()
    test_df[f'{col}_len'] = test_df[col].str.len()

extra_feats = [f'{col}_len' for col in text_cols]
X_train_extra = train_df[extra_feats].fillna(0).values
X_test_extra = test_df[extra_feats].fillna(0).values

# Combine TF-IDF with Extra Features
X_train_final = np.hstack([X_train_tfidf, X_train_extra]) # 5000 + 7 = 5007 features
X_test_final = np.hstack([X_test_tfidf, X_test_extra])   # 5000 + 7 = 5007 features

# Target Variable
y = train_df['Importance Score']

# --- 4. Train-Test Split ---
# For model evaluation
Xtr, Xte, ytr, yte = train_test_split(X_train_final, y, test_size=0.2, random_state=42)

# --- 5. Train the First XGBoost Model ---
# This model gave the lowest RMSE (approx 4.6739)
model_xgb = xgb.XGBRegressor(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
print("First XGBoost model (model_xgb) training is starting...")
model_xgb.fit(Xtr, ytr)
preds_xgb_eval = model_xgb.predict(Xte)
rmse_xgb_eval = np.sqrt(mean_squared_error(yte, preds_xgb_eval))
print(f"\nXGBoost (model_xgb) RMSE (on eval set): {rmse_xgb_eval:.4f}")

# --- 6. Train the Second XGBoost Model (Anti-Overfit) ---
# This was tuned to reduce overfitting
model_anti_over = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.03, max_depth=4,  # Conservative
    subsample=0.75, colsample_bytree=0.75,             # More random
    reg_alpha=0.8, reg_lambda=3.0,                     # Strong reg
    random_state=42, n_jobs=-1
)
print("Anti-Overfit model (model_anti_over) training is starting...")
model_anti_over.fit(Xtr, ytr, eval_set=[(Xte, yte)], verbose=0)
anti_rmse = np.sqrt(mean_squared_error(yte, model_anti_over.predict(Xte)))
print(f"Anti-Overfit XGBoost RMSE (on eval set): {anti_rmse:.4f}")

# --- 7. Ensemble Predictions and RMSE ---
# Ensemble RMSE on evaluation set
final_pred_eval = 0.7 * model_xgb.predict(Xte) + 0.3 * model_anti_over.predict(Xte)
ensemble_rmse_eval = np.sqrt(mean_squared_error(yte, final_pred_eval))
print(f"Ensemble RMSE (on eval set - 0.7 * model_xgb + 0.3 * model_anti_over): {ensemble_rmse_eval:.4f}")

# Final Ensemble Predictions on FULL TEST DATA
xgb_preds_final = model_xgb.predict(X_test_final)
anti_over_preds_final = model_anti_over.predict(X_test_final)
Final_Pred_Submission = 0.7 * xgb_preds_final + 0.3 * anti_over_preds_final

# --- 8. Create Submission File ---
submission_df = pd.DataFrame({'id': test_df['id'], 'Importance Score': Final_Pred_Submission})

# Ensure 'Importance Score' is numeric and handle non-finite values
submission_df['Importance Score'] = pd.to_numeric(submission_df['Importance Score'], errors='coerce')
submission_df['Importance Score'] = submission_df['Importance Score'].fillna(submission_df['Importance Score'].mean())

# Clip negative scores to 0 and max to 92
submission_df['Importance Score'] = submission_df['Importance Score'].apply(lambda x: max(0, x)).clip(0, 92)

submission_df.to_csv('final_top_ensemble_submission.csv', index=False)
print("\nDone! 'final_top_ensemble_submission.csv' is ready for download.")

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [3]:
# Best Performing Model Code

import xgboost as xgb
import lightgbm as lgb # LightGBM was not used in the Ensemble but was part of model development.
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import re

# --- 0. Load Data ---
# Ensure 'train.csv' and 'test.csv' files are available in your /content/ directory.
# If they are not already loaded, load them here.
try:
    train_df = train_df
    test_df = test_df
except NameError:
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

# --- 1. Advanced Text Cleaning & Mega Text Creation ---
def clean_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
    return text

text_cols = ['Headline', 'Lead Types', 'Power Mentions', 'Agencies', 'Reasoning', 'Key Insights', 'Tags']
for col in text_cols:
    train_df[col] = train_df[col].fillna('missing').apply(clean_text)
    test_df[col] = test_df[col].fillna('missing').apply(clean_text)

train_df['mega_text'] = train_df[text_cols].astype(str).agg(' '.join, axis=1)
test_df['mega_text'] = test_df[text_cols].astype(str).agg(' '.join, axis=1)

# --- 2. TF-IDF Vectorization ---
vec = TfidfVectorizer(max_features=5000, ngram_range=(1,3), analyzer='char_wb', max_df=0.95, min_df=2, sublinear_tf=True)
X_train_tfidf = vec.fit_transform(train_df['mega_text']).toarray()
X_test_tfidf = vec.transform(test_df['mega_text']).toarray()

# --- 3. Extra Numeric Features ---
for col in text_cols:
    train_df[f'{col}_len'] = train_df[col].str.len()
    test_df[f'{col}_len'] = test_df[col].str.len()

extra_feats = [f'{col}_len' for col in text_cols]
X_train_extra = train_df[extra_feats].fillna(0).values
X_test_extra = test_df[extra_feats].fillna(0).values

# Combine TF-IDF with Extra Features
X_train_final = np.hstack([X_train_tfidf, X_train_extra]) # 5000 + 7 = 5007 features
X_test_final = np.hstack([X_test_tfidf, X_test_extra])   # 5000 + 7 = 5007 features

# Target Variable
y = train_df['Importance Score']

# --- 4. Train-Test Split ---
# For model evaluation
Xtr, Xte, ytr, yte = train_test_split(X_train_final, y, test_size=0.2, random_state=42)

# --- 5. Train the First XGBoost Model ---
# This model gave the lowest RMSE (approx 4.6739)
model_xgb = xgb.XGBRegressor(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
print("First XGBoost model (model_xgb) training is starting...")
model_xgb.fit(Xtr, ytr)
preds_xgb_eval = model_xgb.predict(Xte)
rmse_xgb_eval = np.sqrt(mean_squared_error(yte, preds_xgb_eval))
print(f"\nXGBoost (model_xgb) RMSE (on eval set): {rmse_xgb_eval:.4f}")

# --- 6. Train the Second XGBoost Model (Anti-Overfit) ---
# This was tuned to reduce overfitting
model_anti_over = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.03, max_depth=4,  # Conservative
    subsample=0.75, colsample_bytree=0.75,             # More random
    reg_alpha=0.8, reg_lambda=3.0,                     # Strong reg
    random_state=42, n_jobs=-1
)
print("Anti-Overfit model (model_anti_over) training is starting...")
model_anti_over.fit(Xtr, ytr, eval_set=[(Xte, yte)], verbose=0)
anti_rmse = np.sqrt(mean_squared_error(yte, model_anti_over.predict(Xte)))
print(f"Anti-Overfit XGBoost RMSE (on eval set): {anti_rmse:.4f}")

# --- 7. Ensemble Predictions and RMSE ---
# Ensemble RMSE on evaluation set
final_pred_eval = 0.7 * model_xgb.predict(Xte) + 0.3 * model_anti_over.predict(Xte)
ensemble_rmse_eval = np.sqrt(mean_squared_error(yte, final_pred_eval))
print(f"Ensemble RMSE (on eval set - 0.7 * model_xgb + 0.3 * model_anti_over): {ensemble_rmse_eval:.4f}")

# Final Ensemble Predictions on FULL TEST DATA
xgb_preds_final = model_xgb.predict(X_test_final)
anti_over_preds_final = model_anti_over.predict(X_test_final)
Final_Pred_Submission = 0.7 * xgb_preds_final + 0.3 * anti_over_preds_final

# --- 8. Create Submission File ---
submission_df = pd.DataFrame({'id': test_df['id'], 'Importance Score': Final_Pred_Submission})

# Ensure 'Importance Score' is numeric and handle non-finite values
submission_df['Importance Score'] = pd.to_numeric(submission_df['Importance Score'], errors='coerce')
submission_df['Importance Score'] = submission_df['Importance Score'].fillna(submission_df['Importance Score'].mean())

# Clip negative scores to 0 and max to 92
submission_df['Importance Score'] = submission_df['Importance Score'].apply(lambda x: max(0, x)).clip(0, 92)

submission_df.to_csv('final_top_ensemble_submission.csv', index=False)
print("\nDone! 'final_top_ensemble_submission.csv' is ready for download.")

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [4]:
# Best Performing Model Code

import xgboost as xgb
import lightgbm as lgb # LightGBM was not used in the Ensemble but was part of model development.
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import re

# --- 0. Load Data ---
# Ensure 'train.csv' and 'test.csv' files are available in your /content/ directory.
# If they are not already loaded, load them here.
try:
    train_df = train_df
    test_df = test_df
except NameError:
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

# --- 1. Advanced Text Cleaning & Mega Text Creation ---
def clean_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
    return text

text_cols = ['Headline', 'Lead Types', 'Power Mentions', 'Agencies', 'Reasoning', 'Key Insights', 'Tags']
for col in text_cols:
    train_df[col] = train_df[col].fillna('missing').apply(clean_text)
    test_df[col] = test_df[col].fillna('missing').apply(clean_text)

train_df['mega_text'] = train_df[text_cols].astype(str).agg(' '.join, axis=1)
test_df['mega_text'] = test_df[text_cols].astype(str).agg(' '.join, axis=1)

# --- 2. TF-IDF Vectorization ---
vec = TfidfVectorizer(max_features=5000, ngram_range=(1,3), analyzer='char_wb', max_df=0.95, min_df=2, sublinear_tf=True)
X_train_tfidf = vec.fit_transform(train_df['mega_text']).toarray()
X_test_tfidf = vec.transform(test_df['mega_text']).toarray()

# --- 3. Extra Numeric Features ---
for col in text_cols:
    train_df[f'{col}_len'] = train_df[col].str.len()
    test_df[f'{col}_len'] = test_df[col].str.len()

extra_feats = [f'{col}_len' for col in text_cols]
X_train_extra = train_df[extra_feats].fillna(0).values
X_test_extra = test_df[extra_feats].fillna(0).values

# Combine TF-IDF with Extra Features
X_train_final = np.hstack([X_train_tfidf, X_train_extra]) # 5000 + 7 = 5007 features
X_test_final = np.hstack([X_test_tfidf, X_test_extra])   # 5000 + 7 = 5007 features

# Target Variable
y = train_df['Importance Score']

# --- 4. Train-Test Split ---
# For model evaluation
Xtr, Xte, ytr, yte = train_test_split(X_train_final, y, test_size=0.2, random_state=42)

# --- 5. Train the First XGBoost Model ---
# This model gave the lowest RMSE (approx 4.6739)
model_xgb = xgb.XGBRegressor(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
print("First XGBoost model (model_xgb) training is starting...")
model_xgb.fit(Xtr, ytr)
preds_xgb_eval = model_xgb.predict(Xte)
rmse_xgb_eval = np.sqrt(mean_squared_error(yte, preds_xgb_eval))
print(f"\nXGBoost (model_xgb) RMSE (on eval set): {rmse_xgb_eval:.4f}")

# --- 6. Train the Second XGBoost Model (Anti-Overfit) ---
# This was tuned to reduce overfitting
model_anti_over = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.03, max_depth=4,  # Conservative
    subsample=0.75, colsample_bytree=0.75,             # More random
    reg_alpha=0.8, reg_lambda=3.0,                     # Strong reg
    random_state=42, n_jobs=-1
)
print("Anti-Overfit model (model_anti_over) training is starting...")
model_anti_over.fit(Xtr, ytr, eval_set=[(Xte, yte)], verbose=0)
anti_rmse = np.sqrt(mean_squared_error(yte, model_anti_over.predict(Xte)))
print(f"Anti-Overfit XGBoost RMSE (on eval set): {anti_rmse:.4f}")

# --- 7. Ensemble Predictions and RMSE ---
# Ensemble RMSE on evaluation set
final_pred_eval = 0.7 * model_xgb.predict(Xte) + 0.3 * model_anti_over.predict(Xte)
ensemble_rmse_eval = np.sqrt(mean_squared_error(yte, final_pred_eval))
print(f"Ensemble RMSE (on eval set - 0.7 * model_xgb + 0.3 * model_anti_over): {ensemble_rmse_eval:.4f}")

# Final Ensemble Predictions on FULL TEST DATA
xgb_preds_final = model_xgb.predict(X_test_final)
anti_over_preds_final = model_anti_over.predict(X_test_final)
Final_Pred_Submission = 0.7 * xgb_preds_final + 0.3 * anti_over_preds_final

# --- 8. Create Submission File ---
submission_df = pd.DataFrame({'id': test_df['id'], 'Importance Score': Final_Pred_Submission})

# Ensure 'Importance Score' is numeric and handle non-finite values
submission_df['Importance Score'] = pd.to_numeric(submission_df['Importance Score'], errors='coerce')
submission_df['Importance Score'] = submission_df['Importance Score'].fillna(submission_df['Importance Score'].mean())

# Clip negative scores to 0 and max to 92
submission_df['Importance Score'] = submission_df['Importance Score'].apply(lambda x: max(0, x)).clip(0, 92)

submission_df.to_csv('final_top_ensemble_submission.csv', index=False)
print("\nDone! 'final_top_ensemble_submission.csv' is ready for download.")

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [5]:
(Best Performing Model Code)

import xgboost as xgb
import lightgbm as lgb # LightGBM was not used in the Ensemble but was part of model development.
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import re

# --- 0. Load Data ---
# Ensure 'train.csv' and 'test.csv' files are available in your /content/ directory.
# If they are not already loaded, load them here.
try:
    train_df = train_df
    test_df = test_df
except NameError:
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

# --- 1. Advanced Text Cleaning & Mega Text Creation ---
def clean_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
    return text

text_cols = ['Headline', 'Lead Types', 'Power Mentions', 'Agencies', 'Reasoning', 'Key Insights', 'Tags']
for col in text_cols:
    train_df[col] = train_df[col].fillna('missing').apply(clean_text)
    test_df[col] = test_df[col].fillna('missing').apply(clean_text)

train_df['mega_text'] = train_df[text_cols].astype(str).agg(' '.join, axis=1)
test_df['mega_text'] = test_df[text_cols].astype(str).agg(' '.join, axis=1)

# --- 2. TF-IDF Vectorization ---
vec = TfidfVectorizer(max_features=5000, ngram_range=(1,3), analyzer='char_wb', max_df=0.95, min_df=2, sublinear_tf=True)
X_train_tfidf = vec.fit_transform(train_df['mega_text']).toarray()
X_test_tfidf = vec.transform(test_df['mega_text']).toarray()

# --- 3. Extra Numeric Features ---
for col in text_cols:
    train_df[f'{col}_len'] = train_df[col].str.len()
    test_df[f'{col}_len'] = test_df[col].str.len()

extra_feats = [f'{col}_len' for col in text_cols]
X_train_extra = train_df[extra_feats].fillna(0).values
X_test_extra = test_df[extra_feats].fillna(0).values

# Combine TF-IDF with Extra Features
X_train_final = np.hstack([X_train_tfidf, X_train_extra]) # 5000 + 7 = 5007 features
X_test_final = np.hstack([X_test_tfidf, X_test_extra])   # 5000 + 7 = 5007 features

# Target Variable
y = train_df['Importance Score']

# --- 4. Train-Test Split ---
# For model evaluation
Xtr, Xte, ytr, yte = train_test_split(X_train_final, y, test_size=0.2, random_state=42)

# --- 5. Train the First XGBoost Model ---
# This model gave the lowest RMSE (approx 4.6739)
model_xgb = xgb.XGBRegressor(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
print("First XGBoost model (model_xgb) training is starting...")
model_xgb.fit(Xtr, ytr)
preds_xgb_eval = model_xgb.predict(Xte)
rmse_xgb_eval = np.sqrt(mean_squared_error(yte, preds_xgb_eval))
print(f"\nXGBoost (model_xgb) RMSE (on eval set): {rmse_xgb_eval:.4f}")

# --- 6. Train the Second XGBoost Model (Anti-Overfit) ---
# This was tuned to reduce overfitting
model_anti_over = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.03, max_depth=4,  # Conservative
    subsample=0.75, colsample_bytree=0.75,             # More random
    reg_alpha=0.8, reg_lambda=3.0,                     # Strong reg
    random_state=42, n_jobs=-1
)
print("Anti-Overfit model (model_anti_over) training is starting...")
model_anti_over.fit(Xtr, ytr, eval_set=[(Xte, yte)], verbose=0)
anti_rmse = np.sqrt(mean_squared_error(yte, model_anti_over.predict(Xte)))
print(f"Anti-Overfit XGBoost RMSE (on eval set): {anti_rmse:.4f}")

# --- 7. Ensemble Predictions and RMSE ---
# Ensemble RMSE on evaluation set
final_pred_eval = 0.7 * model_xgb.predict(Xte) + 0.3 * model_anti_over.predict(Xte)
ensemble_rmse_eval = np.sqrt(mean_squared_error(yte, final_pred_eval))
print(f"Ensemble RMSE (on eval set - 0.7 * model_xgb + 0.3 * model_anti_over): {ensemble_rmse_eval:.4f}")

# Final Ensemble Predictions on FULL TEST DATA
xgb_preds_final = model_xgb.predict(X_test_final)
anti_over_preds_final = model_anti_over.predict(X_test_final)
Final_Pred_Submission = 0.7 * xgb_preds_final + 0.3 * anti_over_preds_final

# --- 8. Create Submission File ---
submission_df = pd.DataFrame({'id': test_df['id'], 'Importance Score': Final_Pred_Submission})

# Ensure 'Importance Score' is numeric and handle non-finite values
submission_df['Importance Score'] = pd.to_numeric(submission_df['Importance Score'], errors='coerce')
submission_df['Importance Score'] = submission_df['Importance Score'].fillna(submission_df['Importance Score'].mean())

# Clip negative scores to 0 and max to 92
submission_df['Importance Score'] = submission_df['Importance Score'].apply(lambda x: max(0, x)).clip(0, 92)

submission_df.to_csv('final_top_ensemble_submission.csv', index=False)
print("\nDone! 'final_top_ensemble_submission.csv' is ready for download.")

SyntaxError: invalid syntax. Perhaps you forgot a comma? (ipython-input-1975682011.py, line 1)

In [ ]:
!ls -F

'bash-8-0-round-2 (1).zip'   sample_data/	       submission_super.csv
 drive/			     submission.csv	       test.csv
 final_top.csv		     submission_ensemble.csv   train.csv


In [6]:
# Best Performing Model Code

import xgboost as xgb
import lightgbm as lgb # LightGBM was not used in the Ensemble but was part of model development.
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import re

# --- 0. Load Data ---
# Ensure 'train.csv' and 'test.csv' files are available in your /content/ directory.
# If they are not already loaded, load them here.
try:
    train_df = train_df
    test_df = test_df
except NameError:
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

# --- 1. Advanced Text Cleaning & Mega Text Creation ---
def clean_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
    return text

text_cols = ['Headline', 'Lead Types', 'Power Mentions', 'Agencies', 'Reasoning', 'Key Insights', 'Tags']
for col in text_cols:
    train_df[col] = train_df[col].fillna('missing').apply(clean_text)
    test_df[col] = test_df[col].fillna('missing').apply(clean_text)

train_df['mega_text'] = train_df[text_cols].astype(str).agg(' '.join, axis=1)
test_df['mega_text'] = test_df[text_cols].astype(str).agg(' '.join, axis=1)

# --- 2. TF-IDF Vectorization ---
vec = TfidfVectorizer(max_features=5000, ngram_range=(1,3), analyzer='char_wb', max_df=0.95, min_df=2, sublinear_tf=True)
X_train_tfidf = vec.fit_transform(train_df['mega_text']).toarray()
X_test_tfidf = vec.transform(test_df['mega_text']).toarray()

# --- 3. Extra Numeric Features ---
for col in text_cols:
    train_df[f'{col}_len'] = train_df[col].str.len()
    test_df[f'{col}_len'] = test_df[col].str.len()

extra_feats = [f'{col}_len' for col in text_cols]
X_train_extra = train_df[extra_feats].fillna(0).values
X_test_extra = test_df[extra_feats].fillna(0).values

# Combine TF-IDF with Extra Features
X_train_final = np.hstack([X_train_tfidf, X_train_extra]) # 5000 + 7 = 5007 features
X_test_final = np.hstack([X_test_tfidf, X_test_extra])   # 5000 + 7 = 5007 features

# Target Variable
y = train_df['Importance Score']

# --- 4. Train-Test Split ---
# For model evaluation
Xtr, Xte, ytr, yte = train_test_split(X_train_final, y, test_size=0.2, random_state=42)

# --- 5. Train the First XGBoost Model ---
# This model gave the lowest RMSE (approx 4.6739)
model_xgb = xgb.XGBRegressor(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
print("First XGBoost model (model_xgb) training is starting...")
model_xgb.fit(Xtr, ytr)
preds_xgb_eval = model_xgb.predict(Xte)
rmse_xgb_eval = np.sqrt(mean_squared_error(yte, preds_xgb_eval))
print(f"\nXGBoost (model_xgb) RMSE (on eval set): {rmse_xgb_eval:.4f}")

# --- 6. Train the Second XGBoost Model (Anti-Overfit) ---
# This was tuned to reduce overfitting
model_anti_over = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.03, max_depth=4,  # Conservative
    subsample=0.75, colsample_bytree=0.75,             # More random
    reg_alpha=0.8, reg_lambda=3.0,                     # Strong reg
    random_state=42, n_jobs=-1
)
print("Anti-Overfit model (model_anti_over) training is starting...")
model_anti_over.fit(Xtr, ytr, eval_set=[(Xte, yte)], verbose=0)
anti_rmse = np.sqrt(mean_squared_error(yte, model_anti_over.predict(Xte)))
print(f"Anti-Overfit XGBoost RMSE (on eval set): {anti_rmse:.4f}")

# --- 7. Ensemble Predictions and RMSE ---
# Ensemble RMSE on evaluation set
final_pred_eval = 0.7 * model_xgb.predict(Xte) + 0.3 * model_anti_over.predict(Xte)
ensemble_rmse_eval = np.sqrt(mean_squared_error(yte, final_pred_eval))
print(f"Ensemble RMSE (on eval set - 0.7 * model_xgb + 0.3 * model_anti_over): {ensemble_rmse_eval:.4f}")

# Final Ensemble Predictions on FULL TEST DATA
xgb_preds_final = model_xgb.predict(X_test_final)
anti_over_preds_final = model_anti_over.predict(X_test_final)
Final_Pred_Submission = 0.7 * xgb_preds_final + 0.3 * anti_over_preds_final

# --- 8. Create Submission File ---
submission_df = pd.DataFrame({'id': test_df['id'], 'Importance Score': Final_Pred_Submission})

# Ensure 'Importance Score' is numeric and handle non-finite values
submission_df['Importance Score'] = pd.to_numeric(submission_df['Importance Score'], errors='coerce')
submission_df['Importance Score'] = submission_df['Importance Score'].fillna(submission_df['Importance Score'].mean())

# Clip negative scores to 0 and max to 92
submission_df['Importance Score'] = submission_df['Importance Score'].apply(lambda x: max(0, x)).clip(0, 92)

submission_df.to_csv('final_top_ensemble_submission.csv', index=False)
print("\nDone! 'final_top_ensemble_submission.csv' is ready for download.")

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
import numpy as np
import xgboost as xgb # Ensure xgboost is imported

groups = train_df['Source File']  # grouping key

gkf = GroupKFold(n_splits=5)

rmse_scores = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X_train_final, y, groups)):
    X_tr, X_te = X_train_final[train_idx], X_train_final[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

    model = xgb.XGBRegressor(
        n_estimators=600,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    rmse = np.sqrt(mean_squared_error(y_te, preds))

    rmse_scores.append(rmse)
    print(f"Fold {fold+1} RMSE: {rmse:.4f}")

print("\nGroupKFold Mean RMSE:", np.mean(rmse_scores))

KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Assuming 'submission' DataFrame is available from previous steps
# If you meant 'submission_df' from the last ensemble, I will use that.
# If you want to use a different submission dataframe, please specify.

# Using 'submission_df' which was created from the best ensemble model
submission_df.to_csv('/content/drive/MyDrive/submission.csv', index=False)
print("Done! 'submission.csv' आपके Google Drive में सेव हो गई है।")

In [ ]:
from google.colab import files

print("Please upload the 'bash-8-0-round-2 (1).zip' file.")
uploaded = files.upload()

# Check if file was uploaded
if 'bash-8-0-round-2 (1).zip' in uploaded:
    print("File uploaded successfully!")
    # Now unzip the zip file
    !mv "/content/bash-8-0-round-2 \(1\).zip" "/content/data.zip"
    !unzip -o "/content/data.zip"
    print("Zip file unzipped. Now try to load train and test data.")
else:
    print("File not uploaded or its name is not as expected.")

Please upload the 'bash-8-0-round-2 (1).zip' file.


In [ ]:
import pandas as pd

# Rename the zip file using backslash escaping for parentheses, then to a simpler name
# This is a more robust way to handle special characters in shell commands.
# यह सेल तभी चलना चाहिए जब फ़ाइल c6a56b36 में सफलतापूर्वक अपलोड और अनज़िप हो जाए

# इस सेल को खाली कर दिया गया है क्योंकि अपलोड और अनज़िप ऑपरेशन c6a56b36 में चले गए हैं।


mv: cannot stat '/content/bash-8-0-round-2 \(1\).zip': No such file or directory
unzip:  cannot find or open /content/data.zip, /content/data.zip.zip or /content/data.zip.ZIP.
Zip file renamed and unzipped. Checking for train.csv and test.csv...


In [ ]:
print("Minimum Score:", train_df['Importance Score'].min())
print("Maximum Score:", train_df['Importance Score'].max())
print("Average Score:", train_df['Importance Score'].mean())

Minimum Score: 0
Maximum Score: 92
Average Score: 18.049214507370053


In [7]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

# Step 1: Combine Text columns (for both Training and Testing)
# We are combining Headline and Key Insights so that the model gets more context
train_df['combined_text'] = train_df['Headline'].astype(str) + ' ' + train_df['Key Insights'].astype(str)
test_df['combined_text'] = test_df['Headline'].astype(str) + ' ' + test_df['Key Insights'].astype(str)

# Step 2: TF-IDF Vectorization (Converting text to numbers)
vec = TfidfVectorizer(max_features=1500, ngram_range=(1,2), stop_words='english')
X = vec.fit_transform(train_df['combined_text']).toarray()
y = train_df['Importance Score']

# Step 3: Train-Test Split (80% training, 20% testing)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 4: XGBoost Model Setup
model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1
)

print("Model training is starting...")
model.fit(X_tr, y_tr)

# Step 5: Score check (RMSE)
preds_te = model.predict(X_te)
rmse = np.sqrt(mean_squared_error(y_te, preds_te))
print(f"\n--- Model Performance ---")
print(f"XGBoost RMSE: {rmse:.4f}")

# Step 6: Prediction on test data and creating Submission file
print("\nCreating final submission file...")
X_test_final = vec.transform(test_df['combined_text']).toarray()
test_df['Importance Score'] = model.predict(X_test_final)

# Save submission file
test_df[['id', 'Importance Score']].to_csv('submission.csv', index=False)
print("Done! 'submission.csv' is ready for download.")

NameError: name 'train_df' is not defined

In [ ]:
from google.colab import files
files.download('submission_ensemble.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
train_df.fillna('missing', inplace=True)
train_df['text'] = train_df['Headline'] + ' ' + train_df['Key Insights']
vec = TfidfVectorizer(max_features=500)
X = vec.fit_transform(train_df['text']).toarray()
y = train_df['Importance Score']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2)
model = RandomForestRegressor(n_estimators=50)
model.fit(X_tr, y_tr)
rmse = np.sqrt(mean_squared_error(y_te, model.predict(X_te)))
print("RMSE:", rmse)

RMSE: 9.716834365251822


In [ ]:
print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)

print("\nTrain Info:")
train_df.info()

# Missing values check karne ke liye
print("\nMissing Values in Train:\n", train_df.isnull().sum())

print("\nTarget col?", [col for col in train_df.columns if 'score' in col.lower() or 'import' in col.lower() or 'target' in col.lower()])

print("\nHead:\n", train_df.head())

अब जबकि `train.csv` और `test.csv` फाइलें उपलब्ध हैं, तो चलिए इन्हें पांडास डेटाफ्रेम में लोड करते हैं।

In [15]:
import pandas as pd

# Load train.csv file
train_df = pd.read_csv('train.csv')

# Load test.csv file
test_df = pd.read_csv('test.csv')

print("First 5 rows of train DataFrame:")
display(train_df.head())

print("First 5 rows of test DataFrame:")
display(test_df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:
# Super Strong Version - RMSE <7 target (XGBoost + Extra Features + Tuning)
import xgboost as xgb; from sklearn.feature_extraction.text import TfidfVectorizer; from sklearn.model_selection import train_test_split; from sklearn.metrics import mean_squared_error; from sklearn.preprocessing import LabelEncoder; import numpy as np; import re

# 1. Advanced Text Cleaning + Mega Text (all columns)
def clean_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
    return text

text_cols = ['Headline', 'Lead Types', 'Power Mentions', 'Agencies', 'Reasoning', 'Key Insights', 'Tags']
for col in text_cols:
    train_df[col] = train_df[col].fillna('missing').apply(clean_text)
    test_df[col] = test_df[col].fillna('missing').apply(clean_text)

train_df['mega_text'] = train_df[text_cols].astype(str).agg(' '.join, axis=1)
test_df['mega_text'] = test_df[text_cols].astype(str).agg(' '.join, axis=1)

# 2. Better TF-IDF (5000 feats, char+word ngrams)
vec = TfidfVectorizer(max_features=5000, ngram_range=(1,3), analyzer='char_wb', max_df=0.95, min_df=2, sublinear_tf=True)
X = vec.fit_transform(train_df['mega_text']).toarray()
X_test_final = vec.transform(test_df['mega_text']).toarray()

# 3. Extra Features (lengths, counts)
for col in text_cols:
    train_df[f'{col}_len'] = train_df[col].str.len()
    test_df[f'{col}_len'] = test_df[col].str.len()
extra_feats = [f'{col}_len' for col in text_cols]
X_extra = train_df[extra_feats].fillna(0).values
X_test_extra = test_df[extra_feats].fillna(0).values
X = np.hstack([X, X_extra])
X_test_final = np.hstack([X_test_final, X_test_extra])

y = train_df['Importance Score']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Tuned XGBoost (best for text regression)
model = xgb.XGBRegressor(n_estimators=300, learning_rate=0.02, max_depth=8, subsample=0.8, colsample_bytree=0.8,
                         reg_alpha=0.1, reg_lambda=1.0, random_state=42, n_jobs=-1)
print("Model training shuru ho rahi hai...")
model.fit(X_tr, y_tr)

# Step 5: Score check (RMSE)
preds = model.predict(X_te)
rmse = np.sqrt(mean_squared_error(y_te, preds))
print(f"\n--- Model Performance ---")
print(f"🚀 Super XGBoost RMSE: {rmse:.4f} (LB top potential!)")

# 6. Submission
print("\nFinal submission file bana rahe hain...")
test_df['Importance Score'] = model.predict(X_test_final)
test_df[['id', 'Importance Score']].to_csv('submission_super.csv', index=False)
print("✅ submission_super.csv ready - Upload to leaderboard!")

In [ ]:
!pip install xgboost -q  # Fast install if missing
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
train_df = pd.read_csv('train.csv').fillna('missing')
test_df = pd.read_csv('test.csv').fillna('missing')
train_df['text'] = train_df['Headline'] + ' ' + train_df['Key Insights']
test_df['text'] = test_df['Headline'] + ' ' + test_df['Key Insights']
vec = TfidfVectorizer(max_features=1000)
X = vec.fit_transform(train_df['text']).toarray()
Xtest = vec.transform(test_df['text']).toarray()
Xtr,Xte,ytr,yte = train_test_split(X,train_df['Importance Score'],test_size=0.2)
model = XGBRegressor(n_estimators=100)
model.fit(Xtr,ytr)
print(np.sqrt(mean_squared_error(yte,model.predict(Xte))))
test_df['Importance Score'] = model.predict(Xtest)
test_df[['id','Importance Score']].to_csv('submission.csv',index=False)

In [8]:
# Super Strong Version - RMSE <7 target (XGBoost + Extra Features + Tuning)
import xgboost as xgb; from sklearn.feature_extraction.text import TfidfVectorizer; from sklearn.model_selection import train_test_split; from sklearn.metrics import mean_squared_error; from sklearn.preprocessing import LabelEncoder; import numpy as np; import re

# 1. Advanced Text Cleaning + Mega Text (all columns)
def clean_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
    return text

text_cols = ['Headline', 'Lead Types', 'Power Mentions', 'Agencies', 'Reasoning', 'Key Insights', 'Tags']
for col in text_cols:
    train_df[col] = train_df[col].fillna('missing').apply(clean_text)
    test_df[col] = test_df[col].fillna('missing').apply(clean_text)

train_df['mega_text'] = train_df[text_cols].astype(str).agg(' '.join, axis=1)
test_df['mega_text'] = test_df[text_cols].astype(str).agg(' '.join, axis=1)

# 2. Better TF-IDF (5000 feats, char+word ngrams)
vec = TfidfVectorizer(max_features=5000, ngram_range=(1,3), analyzer='char_wb', max_df=0.95, min_df=2, sublinear_tf=True)
X = vec.fit_transform(train_df['mega_text']).toarray()
X_test_final = vec.transform(test_df['mega_text']).toarray()

# 3. Extra Features (lengths, counts)
for col in text_cols:
    train_df[f'{col}_len'] = train_df[col].str.len()
    test_df[f'{col}_len'] = test_df[col].str.len()
extra_feats = [f'{col}_len' for col in text_cols]
X_extra = train_df[extra_feats].fillna(0).values
X_test_extra = test_df[extra_feats].fillna(0).values
X = np.hstack([X, X_extra])
X_test_final = np.hstack([X_test_final, X_test_extra])

y = train_df['Importance Score']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Tuned XGBoost (best for text regression)
model = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=8, subsample=0.8, colsample_bytree=0.8,
                         reg_alpha=0.1, reg_lambda=1.0, random_state=42, n_jobs=-1)
print("Model training is starting...")
model.fit(X_tr, y_tr)

# Step 5: Score check (RMSE)
preds = model.predict(X_te)
rmse = np.sqrt(mean_squared_error(y_te, preds))
print(f"\n--- Model Performance ---")
print(f"🚀 Super XGBoost RMSE: {rmse:.4f} (LB top potential!)")

# 6. Submission
print("\nCreating final submission file...")
test_df['Importance Score'] = model.predict(X_test_final)
test_df[['id', 'Importance Score']].to_csv('submission_super.csv', index=False)
print("submission_super.csv ready - Upload to leaderboard!")

NameError: name 'train_df' is not defined

In [ ]:
from sklearn.ensemble import VotingRegressor
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# Do best models ko define karna
xgb_model = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, n_jobs=-1)
rf_model = RandomForestRegressor(n_estimators=200, max_depth=12, n_jobs=-1)

# Dono ko milakar ek "Ensemble" banana
ensemble_model = VotingRegressor(estimators=[
    ('xgb', xgb_model),
    ('rf', rf_model)
])

print("Ensemble model train ho raha hai... isme 5 minute lag sakte hain.")
ensemble_model.fit(Xtr, ytr)

# Accuracy check
ensemble_rmse = np.sqrt(mean_squared_error(yte, ensemble_model.predict(Xte)))
print(f"Ensemble RMSE: {ensemble_rmse:.4f}")

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

model = XGBRegressor(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(Xtr, ytr)

preds = model.predict(Xte)
rmse = np.sqrt(mean_squared_error(yte, preds))

print("Final RMSE:", rmse)

In [ ]:
from sklearn.model_selection import train_test_split

y = train_df['Importance Score']

Xtr, Xte, ytr, yte = train_test_split(
    X_final, y, test_size=0.2, random_state=42
)

In [ ]:
test_df['power_count'] = test_df['Power Mentions'].apply(
    lambda x: len(str(x).split(',')) if x != 'missing' else 0
)

test_df['agency_count'] = test_df['Agencies'].apply(
    lambda x: len(str(x).split(',')) if x != 'missing' else 0
)

test_df['headline_len'] = test_df['Headline'].str.len()
test_df['insights_len'] = test_df['Key Insights'].str.len()

test_extra = test_df[
    ['power_count', 'agency_count', 'headline_len', 'insights_len']
].values

X_test_final = np.hstack([Xtest, test_extra])

In [ ]:
# Extra numeric features
extra_features = train_df[
    ['power_count', 'agency_count', 'headline_len', 'insights_len']
].values

# TF-IDF features (जो आपने पहले बनाए थे)
X_text = X   # assuming X = TF-IDF output (numpy array)

# Combine text + numeric
import numpy as np
X_final = np.hstack([X_text, extra_features])

In [ ]:
train_df['power_count'] = train_df['Power Mentions'].apply(
    lambda x: len(str(x).split(',')) if x != 'missing' else 0
)

train_df['agency_count'] = train_df['Agencies'].apply(
    lambda x: len(str(x).split(',')) if x != 'missing' else 0
)

train_df['headline_len'] = train_df['Headline'].str.len()
train_df['insights_len'] = train_df['Key Insights'].str.len()

In [12]:
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
import numpy as np

model_lgb = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42 # for reproducibility
)

print("LightGBM model training is starting...")
model_lgb.fit(Xtr, ytr)

# Make predictions on the test set
y_pred_lgb = model_lgb.predict(Xte)

# Calculate RMSE
rmse_lgb = np.sqrt(mean_squared_error(yte, y_pred_lgb))
print(f"\nLightGBM RMSE: {rmse_lgb:.4f}")

LightGBM model training is starting...


NameError: name 'Xtr' is not defined

In [ ]:
train_df['power_count'] = train_df['Power Mentions'].apply(lambda x: len(str(x).split(',')))
train_df['agency_count'] = train_df['Agencies'].apply(lambda x: len(str(x).split(',')))

In [ ]:
y_pred = np.expm1(y_pred_log)
rmse_original = np.sqrt(mean_squared_error(yte, y_pred))
print("Original-scale RMSE:", rmse_original)

In [13]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
from sklearn.model_selection import cross_val_score

# New hyperparameters
learning_rate = 0.03
n_estimators = 800
max_depth = 4

# Apply log transformation to the target variable
ytr_log = np.log1p(ytr)
yte_log = np.log1p(yte)

# Redefine the model with new hyperparameters
model_tuned = XGBRegressor(
    n_estimators=n_estimators,
    max_depth=max_depth,
    learning_rate=learning_rate,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=2,
    random_state=42
)

print("Tuned model training is starting...")
model_tuned.fit(Xtr, ytr_log)

# Calculate RMSE directly on the test set
y_pred_log = model_tuned.predict(Xte)
rmse_direct = np.sqrt(mean_squared_error(yte_log, y_pred_log))
print(f"\nDirect test set RMSE (Log Transform): {rmse_direct:.4f}")

# Calculate RMSE using cross-validation
# Note: Using 'ytr' for cross-validation and 'neg_root_mean_squared_error' scoring
# If you want to perform CV on log-transformed y, use 'ytr_log'
scores = cross_val_score(
    model_tuned,
    Xtr,
    ytr_log, # CV on log-transformed y
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1 # for parallel processing
)

print(f"CV RMSE (Log Transform): {-scores.mean():.4f}")

NameError: name 'ytr' is not defined

In [14]:
# Save final predictions to submission file
submission_df = pd.DataFrame({'id': test_df['id'], 'Importance Score': Final_Pred})

# Ensure 'Importance Score' column is of numeric type, handle potential non-finite values
submission_df['Importance Score'] = pd.to_numeric(submission_df['Importance Score'], errors='coerce')
submission_df['Importance Score'] = submission_df['Importance Score'].fillna(submission_df['Importance Score'].mean())

# Clip negative scores to 0
submission_df['Importance Score'] = submission_df['Importance Score'].apply(lambda x: max(0, x))

# Save submission file
submission_df.to_csv('submission_ensemble.csv', index=False)
print("Done! 'submission_ensemble.csv' is ready for download.")

NameError: name 'test_df' is not defined

In [10]:
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import re

# Ensure train_df and test_df are loaded (if not already in current kernel state)
# This is redundant if bc90be93 was the last successful data load, but safe to include.
try:
    train_df = train_df
    test_df = test_df
except NameError:
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

# 1. Advanced Text Cleaning + Mega Text (all columns)
def clean_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
    return text

text_cols = ['Headline', 'Lead Types', 'Power Mentions', 'Agencies', 'Reasoning', 'Key Insights', 'Tags']
for col in text_cols:
    train_df[col] = train_df[col].fillna('missing').apply(clean_text)
    test_df[col] = test_df[col].fillna('missing').apply(clean_text)

train_df['mega_text'] = train_df[text_cols].astype(str).agg(' '.join, axis=1)
test_df['mega_text'] = test_df[text_cols].astype(str).agg(' '.join, axis=1)

# 2. Better TF-IDF (5000 feats, char+word ngrams)
vec = TfidfVectorizer(max_features=5000, ngram_range=(1,3), analyzer='char_wb', max_df=0.95, min_df=2, sublinear_tf=True)
X_train_tfidf = vec.fit_transform(train_df['mega_text']).toarray()
X_test_tfidf = vec.transform(test_df['mega_text']).toarray()

# 3. Extra Features (lengths, counts) - for both train and test
for col in text_cols:
    train_df[f'{col}_len'] = train_df[col].str.len()
    test_df[f'{col}_len'] = test_df[col].str.len()

extra_feats = [f'{col}_len' for col in text_cols]
X_train_extra = train_df[extra_feats].fillna(0).values
X_test_extra = test_df[extra_feats].fillna(0).values

# Combine TF-IDF with extra features
X_train_final = np.hstack([X_train_tfidf, X_train_extra]) # This will be 5000 + len(extra_feats) features
X_test_final = np.hstack([X_test_tfidf, X_test_extra])   # This will be 5000 + len(extra_feats) features


y = train_df['Importance Score']

# 4. Train-Test Split for model evaluation
Xtr, Xte, ytr, yte = train_test_split(X_train_final, y, test_size=0.2, random_state=42)

# 5. Train XGBoost Model
model_xgb = xgb.XGBRegressor(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
print("XGBoost model training is starting...")
model_xgb.fit(Xtr, ytr)
preds_xgb_eval = model_xgb.predict(Xte)
rmse_xgb_eval = np.sqrt(mean_squared_error(yte, preds_xgb_eval))
print(f"\nXGBoost RMSE (on eval set with {Xtr.shape[1]} features): {rmse_xgb_eval:.4f}")

# 6. Train LightGBM Model
model_lgb = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42 # for reproducibility
)
print("LightGBM model training is starting...")
model_lgb.fit(Xtr, ytr)
preds_lgb_eval = model_lgb.predict(Xte)
rmse_lgb_eval = np.sqrt(mean_squared_error(yte, preds_lgb_eval))
print(f"LightGBM RMSE (on eval set with {Xtr.shape[1]} features): {rmse_lgb_eval:.4f}")

# 7. Create Ensemble Predictions for submission on the FULL TEST DATA
print("\nCreating Final Ensemble Predictions...")
xgb_preds_final = model_xgb.predict(X_test_final)
lgbm_preds_final = model_lgb.predict(X_test_final)

Final_Pred = 0.7 * xgb_preds_final + 0.3 * lgbm_preds_final

# 8. Create Submission File
submission_df = pd.DataFrame({'id': test_df['id'], 'Importance Score': Final_Pred})

# Ensure 'Importance Score' column is of numeric type, handle potential non-finite values
submission_df['Importance Score'] = pd.to_numeric(submission_df['Importance Score'], errors='coerce')
submission_df['Importance Score'] = submission_df['Importance Score'].fillna(submission_df['Importance Score'].mean())

# Clip negative scores to 0
submission_df['Importance Score'] = submission_df['Importance Score'].apply(lambda x: max(0, x))

# Save submission file
submission_df.to_csv('submission_ensemble.csv', index=False)
print("Done! 'submission_ensemble.csv' is ready for download.")

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

model = XGBRegressor(
    n_estimators=600,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=2,
    random_state=42
)

model.fit(Xtr, ytr)

y_pred = model.predict(Xte)

rmse = np.sqrt(mean_squared_error(yte, y_pred))
print("RMSE:", rmse)

In [9]:
# Super Strong Version - RMSE <7 target (XGBoost + Extra Features + Tuning)
import xgboost as xgb; from sklearn.feature_extraction.text import TfidfVectorizer; from sklearn.model_selection import train_test_split; from sklearn.metrics import mean_squared_error; from sklearn.preprocessing import LabelEncoder; import numpy as np; import re

# 1. Advanced Text Cleaning + Mega Text (all columns)
def clean_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
    return text

text_cols = ['Headline', 'Lead Types', 'Power Mentions', 'Agencies', 'Reasoning', 'Key Insights', 'Tags']
for col in text_cols:
    train_df[col] = train_df[col].fillna('missing').apply(clean_text)
    test_df[col] = test_df[col].fillna('missing').apply(clean_text)

train_df['mega_text'] = train_df[text_cols].astype(str).agg(' '.join, axis=1)
test_df['mega_text'] = test_df[text_cols].astype(str).agg(' '.join, axis=1)

# 2. Better TF-IDF (5000 feats, char+word ngrams)
vec = TfidfVectorizer(max_features=5000, ngram_range=(1,3), analyzer='char_wb', max_df=0.95, min_df=2, sublinear_tf=True)
X = vec.fit_transform(train_df['mega_text']).toarray()
X_test_final = vec.transform(test_df['mega_text']).toarray()

# 3. Extra Features (lengths, counts)
for col in text_cols:
    train_df[f'{col}_len'] = train_df[col].str.len()
    test_df[f'{col}_len'] = test_df[col].str.len()
extra_feats = [f'{col}_len' for col in text_cols]
X_extra = train_df[extra_feats].fillna(0).values
X_test_extra = test_df[extra_feats].fillna(0).values
X = np.hstack([X, X_extra])
X_test_final = np.hstack([X_test_final, X_test_extra])

y = train_df['Importance Score']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Tuned XGBoost (best for text regression)
model = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=8, subsample=0.8, colsample_bytree=0.8,
                         reg_alpha=0.1, reg_lambda=1.0, random_state=42, n_jobs=-1)
print("Model training is starting...")
model.fit(X_tr, y_tr)

# Step 5: Score check (RMSE)
preds = model.predict(X_te)
rmse = np.sqrt(mean_squared_error(y_te, preds))
print(f"\n--- Model Performance ---")
print(f" Super XGBoost RMSE: {rmse:.4f} (LB top potential!)")

# 6. Submission
print("\nCreating final submission file...")
test_df['Importance Score'] = model.predict(X_test_final)
test_df[['id', 'Importance Score']].to_csv('submission_super.csv', index=False)
print(" submission_super.csv ready - Upload to leaderboard!")

NameError: name 'train_df' is not defined

In [ ]:
from google.colab import files
files.download('submission_ensemble.csv')

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

# XGBoost model से भविष्यवाणियां प्राप्त करें (model_xgb अब सही मॉडल है)
xgb_preds = model_xgb.predict(X_test_final)

# LightGBM model से भविष्यवाणियां प्राप्त करें (model_lgb अब सही मॉडल है)
lgbm_preds = model_lgb.predict(X_test_final)

# भारित औसत का उपयोग करके अंतिम भविष्यवाणी की गणना करें
Final_Pred = 0.7 * xgb_preds + 0.3 * lgbm_preds

# अब RMSE गणना छोड़ दी गई है क्योंकि Final_Pred पूरे टेस्ट सेट के लिए है
# final_ensemble_rmse = np.sqrt(mean_squared_error(yte, Final_Pred))
# print(f"Final Ensemble Prediction RMSE (XGBoost 0.7 + LightGBM 0.3): {final_ensemble_rmse:.4f}")

In [ ]:
pred_tr = model_xgb.predict(Xtr)
pred_te = model_xgb.predict(Xte)

rmse_tr = np.sqrt(mean_squared_error(ytr, pred_tr))
rmse_te = np.sqrt(mean_squared_error(yte, pred_te))

print(f"Training RMSE: {rmse_tr:.4f}")
print(f"Test RMSE: {rmse_te:.4f}")

In [ ]:
from xgboost import XGBRegressor

model_xgb = XGBRegressor(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

print("XGBoost model (model_xgb) training shuru ho rahi hai...")
model_xgb.fit(Xtr, ytr)

In [11]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd # Ensure pandas is imported if not already

# Anti-Overfit Model Setup
model_anti_over = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.03, max_depth=4,  # Conservative
    subsample=0.75, colsample_bytree=0.75,             # More random
    reg_alpha=0.8, reg_lambda=3.0,                     # Strong reg
    random_state=42, n_jobs=-1
)

print("Anti-Overfit model training is starting...")
model_anti_over.fit(Xtr, ytr, eval_set=[(Xte, yte)], verbose=0)

anti_rmse = np.sqrt(mean_squared_error(yte, model_anti_over.predict(Xte)))
print("Anti-Overfit RMSE:", anti_rmse)

# Ensemble (weighted - best for LB)
# Ensure model_xgb is available from previous runs
final_pred = 0.7 * model_xgb.predict(Xte) + 0.3 * model_anti_over.predict(Xte)
print("Ensemble RMSE:", np.sqrt(mean_squared_error(yte, final_pred)))

# Submit
test_df['Importance Score'] = 0.7 * model_xgb.predict(X_test_final) + 0.3 * model_anti_over.predict(X_test_final)
test_df[['id', 'Importance Score']].clip(0,92).to_csv('final_top.csv', index=False)
print("Done! 'final_top.csv' is ready for download.")

Anti-Overfit model training is starting...


NameError: name 'Xtr' is not defined

In [ ]:
from sklearn.model_selection import train_test_split

Xtr, Xte, ytr, yte = train_test_split(
    X_final, y, test_size=0.2, random_state=42
)

# Task
Run cell `c6a56b36` to upload and unzip the `bash-8-0-round-2 (1).zip` file. Please upload the file when prompted.

## फ़ाइलें अपलोड करें और अनज़िप करें

### Subtask:
यह सेल `c6a56b36` चलाएँ ताकि आप अपनी ज़िप फ़ाइल (`bash-8-0-round-2 (1).zip`) अपलोड कर सकें और उसे अनज़िप कर सकें। यह सुनिश्चित करेगा कि `train.csv` और `test.csv` जैसी आवश्यक डेटा फ़ाइलें उपलब्ध हैं।


### Subtask Instructions

1.  **Run cell `c6a56b36`**: यह सेल `c6a56b36` चलाएँ।
2.  **Upload the file**: जब प्रॉम्प्ट किया जाए, तो अपने कंप्यूटर से `bash-8-0-round-2 (1).zip` फ़ाइल अपलोड करें।

यह सुनिश्चित करेगा कि `train.csv` और `test.csv` जैसी आवश्यक डेटा फ़ाइलें आपके कोलाब वातावरण में उपलब्ध हैं।

## डेटा लोड करें

### Subtask:
यह सेल `bc90be93` चलाएँ जो `train.csv` और `test.csv` को पांडास डेटाफ्रेम में लोड करेगा। इससे आगे के विश्लेषण और मॉडल प्रशिक्षण के लिए डेटा तैयार हो जाएगा।


**Reasoning**:
The subtask is to execute cell `bc90be93` which loads `train.csv` and `test.csv` into pandas DataFrames. Executing this cell will prepare the data for further analysis.



In [16]:
import pandas as pd

# Load train.csv file
train_df = pd.read_csv('train.csv')

# Load test.csv file
test_df = pd.read_csv('test.csv')

print("First 5 rows of train DataFrame:")
display(train_df.head())

print("First 5 rows of test DataFrame:")
display(test_df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

**Reasoning**:
The `FileNotFoundError` indicates that `train.csv` and `test.csv` are not available. This is likely because the previous file upload and unzip operation in cell `c6a56b36` failed, as indicated by the `TypeError` in its previous execution. Therefore, I need to guide the user to correctly run `c6a56b36` again to ensure the files are present.



In [ ]:
from google.colab import files

print("Please upload the 'bash-8-0-round-2 (1).zip' file.")
uploaded = files.upload()

# Check if file was uploaded
if 'bash-8-0-round-2 (1).zip' in uploaded:
    print("File uploaded successfully!")
    # Now unzip the zip file
    !mv "/content/bash-8-0-round-2 \(1\).zip" "/content/data.zip"
    !unzip -o "/content/data.zip"
    print("Zip file unzipped. Now try to load train and test data.")
else:
    print("File not uploaded or its name is not as expected.")

माफ़ कीजिए, ऐसा लगता है कि पिछली बार आपने ग़लत फ़ाइल अपलोड की थी।

**कृपया सही फ़ाइल 'bash-8-0-round-2 (1).zip' अपलोड करें।**

यह सुनिश्चित करने के लिए कि 'train.csv' और 'test.csv' फाइलें उपलब्ध हैं, कृपया फिर से **सेल `c6a56b36` चलाएँ** और जब प्रॉम्प्ट किया जाए तो सही ज़िप फ़ाइल अपलोड करें।

माफ़ कीजिए, ऐसा लगता है कि पिछली बार आपने ग़लत फ़ाइल अपलोड की थी।

**कृपया सही फ़ाइल 'bash-8-0-round-2 (1).zip' अपलोड करें।**

यह सुनिश्चित करने के लिए कि 'train.csv' और 'test.csv' फाइलें उपलब्ध हैं, कृपया फिर से **सेल `c6a56b36` चलाएँ** और जब प्रॉम्प्ट किया जाए तो सही ज़िप फ़ाइल अपलोड करें।

**Reasoning**:
The user needs to be explicitly guided to run cell `c6a56b36` again to re-attempt the file upload and unzipping process, as per the instructions in the previous markdown block.



In [ ]:
from google.colab import files

print("Please upload the 'bash-8-0-round-2 (1).zip' file.")
uploaded = files.upload()

# Check if file was uploaded
if 'bash-8-0-round-2 (1).zip' in uploaded:
    print("File uploaded successfully!")
    # Now unzip the zip file
    !mv "/content/bash-8-0-round-2 \(1\).zip" "/content/data.zip"
    !unzip -o "/content/data.zip"
    print("Zip file unzipped. Now try to load train and test data.")
else:
    print("File not uploaded or its name is not as expected.")

Please upload the 'bash-8-0-round-2 (1).zip' file.


**Reasoning**:
The user has provided the next step to re-execute cell `c6a56b36`. The previous output indicates that the cell is executing and waiting for user interaction (file upload). I will provide the expectation for this execution.



In [ ]:
from google.colab import files

print("Please upload the 'bash-8-0-round-2 (1).zip' file.")
uploaded = files.upload()

# Check if file was uploaded
if 'bash-8-0-round-2 (1).zip' in uploaded:
    print("File uploaded successfully!")
    # Now unzip the zip file
    !mv "/content/bash-8-0-round-2 \(1\).zip" "/content/data.zip"
    !unzip -o "/content/data.zip"
    print("Zip file unzipped. Now try to load train and test data.")
else:
    print("File not uploaded or its name is not as expected.")

अब कृपया सही फ़ाइल **`bash-8-0-round-2 (1).zip`** अपलोड करें जब प्रॉम्प्ट किया जाए। यदि अपलोड सफल होता है, तो `train.csv` और `test.csv` फाइलें उपलब्ध होंगी।

**Reasoning**:
The previous step (running cell `c6a56b36`) is waiting for user input to upload the `bash-8-0-round-2 (1).zip` file. After the user successfully uploads the file and it's unzipped, the `train.csv` and `test.csv` files should be available. The next logical step is to attempt loading these files again using cell `bc90be93` as per the subtask instruction.



In [ ]:
import pandas as pd

# Load train.csv file
train_df = pd.read_csv('train.csv')

# Load test.csv file
test_df = pd.read_csv('test.csv')

print("First 5 rows of train DataFrame:")
display(train_df.head())

print("First 5 rows of test DataFrame:")
display(test_df.head())